# Zomato Data Analysis

## Project Overview
This notebook performs exploratory data analysis (EDA) on a cleaned Zomato restaurant dataset.

### Objectives
- Understand restaurant distribution across locations and restaurant types
- Analyze ratings, votes, and cost per person
- Compare online ordering and table-booking availability
- Explore cuisine patterns
- Examine relationships between ratings, votes, and pricing
- Generate visual insights that can support restaurant and location-level business decisions

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("zomato_clean_data.csv")

print("Dataset shape:", df.shape)
df.head()


## 1. Data Overview

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 2. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)


## 3. Restaurant Distribution by Location

In [ ]:
location_counts = (
    df["location"]
    .value_counts()
    .head(15)
    .sort_values()
)

plt.figure(figsize=(10, 6))
location_counts.plot(kind="barh")
plt.title("Top 15 Locations by Number of Restaurants")
plt.xlabel("Number of Restaurants")
plt.ylabel("Location")
plt.tight_layout()
plt.show()


## 4. Restaurant Type Analysis

In [ ]:
type_counts = df["type_of_restaurant"].value_counts().head(15)

plt.figure(figsize=(10, 6))
type_counts.sort_values().plot(kind="barh")
plt.title("Most Common Restaurant Types")
plt.xlabel("Number of Restaurants")
plt.ylabel("Restaurant Type")
plt.tight_layout()
plt.show()


## 5. Rating Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["rating"].dropna(), bins=20, kde=True)
plt.title("Distribution of Restaurant Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Restaurants")
plt.tight_layout()
plt.show()

print("Average rating:", round(df["rating"].mean(), 2))


## 6. Votes and Ratings

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="votes",
    y="rating",
    alpha=0.4
)
plt.title("Votes vs Restaurant Rating")
plt.xlabel("Votes")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

print("Correlation between votes and rating:",
      round(df["votes"].corr(df["rating"]), 3))


## 7. Cost per Person Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["cost_per_person"].dropna(), bins=30, kde=True)
plt.title("Distribution of Cost per Person")
plt.xlabel("Cost per Person")
plt.ylabel("Number of Restaurants")
plt.tight_layout()
plt.show()

print("Average cost per person:",
      round(df["cost_per_person"].mean(), 2))


## 8. Online Order Analysis

In [ ]:
order_summary = (
    df.groupby("online_order")
      .agg(
          restaurant_count=("name", "count"),
          avg_rating=("rating", "mean"),
          avg_cost=("cost_per_person", "mean")
      )
      .round(2)
      .sort_values("restaurant_count", ascending=False)
)

display(order_summary)

plt.figure(figsize=(7, 5))
sns.barplot(data=order_summary.reset_index(),
            x="online_order", y="restaurant_count")
plt.title("Restaurants by Online Order Availability")
plt.xlabel("Online Order")
plt.ylabel("Restaurant Count")
plt.tight_layout()
plt.show()


## 9. Table Booking Analysis

In [ ]:
booking_summary = (
    df.groupby("book_table")
      .agg(
          restaurant_count=("name", "count"),
          avg_rating=("rating", "mean"),
          avg_cost=("cost_per_person", "mean")
      )
      .round(2)
)

display(booking_summary)

plt.figure(figsize=(7, 5))
sns.barplot(data=booking_summary.reset_index(),
            x="book_table", y="restaurant_count")
plt.title("Restaurants by Table Booking Availability")
plt.xlabel("Table Booking")
plt.ylabel("Restaurant Count")
plt.tight_layout()
plt.show()


## 10. Average Rating by Online Order

In [ ]:
rating_order = (
    df.groupby("online_order")["rating"]
      .mean()
      .sort_values(ascending=False)
      .round(2)
)

display(rating_order)

plt.figure(figsize=(7, 5))
rating_order.plot(kind="bar")
plt.title("Average Rating by Online Order Availability")
plt.xlabel("Online Order")
plt.ylabel("Average Rating")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Average Rating by Table Booking

In [ ]:
rating_booking = (
    df.groupby("book_table")["rating"]
      .mean()
      .sort_values(ascending=False)
      .round(2)
)

display(rating_booking)

plt.figure(figsize=(7, 5))
rating_booking.plot(kind="bar")
plt.title("Average Rating by Table Booking Availability")
plt.xlabel("Table Booking")
plt.ylabel("Average Rating")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Top Restaurants by Votes

In [ ]:
top_voted = (
    df[["name", "location", "rating", "votes", "cost_per_person"]]
    .sort_values(["votes", "rating"], ascending=[False, False])
    .head(20)
)

display(top_voted)


## 13. Highly Rated Restaurants

In [ ]:
high_rated = (
    df[df["rating"] >= 4.5]
    [["name", "location", "rating", "votes", "cost_per_person"]]
    .sort_values(["rating", "votes"], ascending=[False, False])
    .head(20)
)

display(high_rated)


## 14. Cuisine Analysis

In [ ]:
# The cuisine field may contain multiple cuisines in one text value.
# Split and explode it to estimate cuisine-level restaurant presence.
cuisine_series = (
    df["cuisines"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

cuisine_counts = cuisine_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
cuisine_counts.sort_values().plot(kind="barh")
plt.title("Top 15 Cuisines by Restaurant Presence")
plt.xlabel("Restaurant Records")
plt.ylabel("Cuisine")
plt.tight_layout()
plt.show()

display(cuisine_counts.to_frame("restaurant_records"))


## 15. Location-level Performance

In [ ]:
location_summary = (
    df.groupby("location")
      .agg(
          restaurant_count=("name", "count"),
          avg_rating=("rating", "mean"),
          avg_votes=("votes", "mean"),
          avg_cost_per_person=("cost_per_person", "mean")
      )
)

location_summary = location_summary[location_summary["restaurant_count"] >= 20]
location_summary = location_summary.sort_values("avg_rating", ascending=False)

display(location_summary.head(15).round(2))


## 16. Cost Segmentation

In [ ]:
df["cost_segment"] = pd.cut(
    df["cost_per_person"],
    bins=[-np.inf, 300, 700, np.inf],
    labels=["Budget", "Mid-Range", "Premium"]
)

cost_segment_summary = (
    df.groupby("cost_segment", observed=False)
      .agg(
          restaurant_count=("name", "count"),
          avg_rating=("rating", "mean"),
          avg_votes=("votes", "mean")
      )
      .round(2)
)

display(cost_segment_summary)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=cost_segment_summary.reset_index(),
    x="cost_segment",
    y="restaurant_count"
)
plt.title("Restaurants by Cost Segment")
plt.xlabel("Cost Segment")
plt.ylabel("Restaurant Count")
plt.tight_layout()
plt.show()


## 17. Key Business Questions

In [ ]:
questions = [
    "Which locations have the largest restaurant presence?",
    "Which restaurant types are most common?",
    "How are ratings distributed across restaurants?",
    "How does online ordering availability compare with ratings and cost?",
    "How does table-booking availability compare with ratings and cost?",
    "Which restaurants receive the highest number of votes?",
    "Which cuisines have the highest representation?",
    "Which locations combine a meaningful restaurant count with stronger average ratings?",
    "How does restaurant performance differ across budget, mid-range, and premium segments?"
]

for i, question in enumerate(questions, 1):
    print(f"{i}. {question}")


## 18. Conclusion

This analysis provides a structured view of restaurant distribution, customer ratings, engagement through votes, pricing, ordering options, table booking, cuisines, and location-level performance.

The findings from this notebook can be used as the analytical foundation for a Power BI dashboard and for the SQL analysis included in this repository.

> Note: Conclusions should be interpreted within the scope of the available cleaned dataset and its variables.
